In [167]:
import torch
import torch.nn as nn

In [168]:
class MultiLatentAttention(nn.Module):
    def __init__(self, embed_dim, latent_dim, num_heads, max_seq_len = 1024, dropout=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.WDKV = nn.Linear(embed_dim, latent_dim)
        self.WDK = nn.Linear(latent_dim, embed_dim)
        self.WDV = nn.Linear(latent_dim, embed_dim)
        self.WQ = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask", torch.triu(torch.ones(1,1, max_seq_len, max_seq_len), diagonal = 1).bool(), persistent = False)
        self.register_buffer("cache_udkv", torch.empty(0), persistent = False)
        self.ptr_current_seq = 0

    def forward(self, X, use_cache = False):
        #CKV Cache implementation temporarily deferred
        batch_size, seq_len, embed_dim = X.size()
        initial_pos = self.ptr_current_seq

        Q = self.WQ(X) #Q = X * WQ
        UDKV = self.WDKV(X) #UDKV = X * WDKV

        if use_cache:
            self.cache_udkv = torch.cat([self.cache_udkv, UDKV], dim = 1)
            UDKV = self.cache_udkv
        self.ptr_current_seq += seq_len

        K = self.WDK(UDKV) #K = UDKV * WDK
        V = self.WDV(UDKV)  #V = UDKV * WDV

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, self.ptr_current_seq, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, self.ptr_current_seq, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5) # attn_scores = Q * K^T / sqrt(embed_dim)
        #print(attn_scores.shape)
        #print(self.mask[initial_pos : initial_pos + seq_len, :initial_pos + seq_len].shape)
        attn_scores = attn_scores.masked_fill(
                        self.mask[:, :, initial_pos : initial_pos + seq_len, :initial_pos + seq_len], float('-inf') #torch.inf * (-1)
                        )
        attn_weights = nn.functional.softmax(attn_scores, dim = -1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, seq_len, embed_dim)
        context = self.out_proj(context)
        return context

In [169]:
X = torch.randn((2,4,8))

In [170]:
attn_layer = MultiLatentAttention(8, 4, 4)
attn_out = attn_layer(X, use_cache = False)

In [171]:
attn_layer = MultiLatentAttention(8, 8, 4)
attn_out = attn_layer(X, use_cache = True)

In [172]:
X1 = torch.randn((2,1,8))
attn_2_out = attn_layer(X1, use_cache = True)

In [173]:
attn_2_out.shape

torch.Size([2, 1, 8])

In [174]:
# Create a 4x4 matrix
A = torch.tensor([[ 1,  2,  3,  4],
                  [ 5,  6,  7,  8],
                  [ 9, 10, 11, 12],
                  [13, 14, 15, 16]])

# Default upper triangular (diagonal=0)
result = torch.triu(A)

In [175]:
a = torch.ones(1,7)
torch.cat([a, torch.empty(0)])

tensor([[1., 1., 1., 1., 1., 1., 1.]])

Testing MLA Implementation

In [176]:
class MultiHeadLatentAttention(nn.Module):
    """
    Implementation of Multi-Head Latent Attention (MLA) as described
    in the DeepSeek architecture. This version focuses on the core
    "compress for storage, decompress for use" mechanism for the
    Key and Value matrices.
    """
    def __init__(self, d_model, num_heads, d_latent, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_latent = d_latent # The dimension of the compressed latent space

        # The Query projection remains standard, projecting to the full model dimension.
        self.W_q = nn.Linear(d_model, d_model)

        # The new KV Down-Projector. This is the "compress" step.
        # It projects the input down to a small, shared latent space.
        self.W_dkv = nn.Linear(d_model, d_latent)

        # The new Key and Value Up-Projectors. This is the "decompress" step.
        # They reconstruct the full-sized K and V from the latent space.
        # Note: These are multi-headed to preserve head diversity.
        self.W_uk = nn.Linear(d_latent, d_model)
        self.W_uv = nn.Linear(d_latent, d_model)

        # The final output projection, standard for multi-head attention.
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        # Causal mask to prevent attending to future tokens. Using a fixed size for demo.
        self.register_buffer('mask', torch.triu(
            torch.ones(1, 1, 1024, 1024), diagonal=1).bool())

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # 1. Query Path (Unchanged)
        # Project and reshape the query as in standard MHA.
        q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # 2. Key/Value Path (The MLA Innovation)
        # Step 2a: Down-Project to the latent space.
        # This is the ONLY value that would be cached during inference.
        c_kv = self.W_dkv(x) # Shape: (batch, seq_len, d_latent)

        # Step 2b: Up-Project from the latent space to get full K and V.
        # These are computed on the fly and are not cached.
        k = self.W_uk(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        v = self.W_uv(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # 3. Standard Attention Calculation
        # The rest of the process is identical to standard MHA.
        attn_scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)

        # Apply causal mask
        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len], float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_weights @ v).transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model)

        # 4. Final Output Projection
        output = self.W_o(context_vector)
        return output

In [177]:
# --- Usage Example ---
d_model = 512
num_heads = 8
d_latent = 128  # Latent dimension must be smaller than d_model
batch_size = 4
seq_len = 64

In [178]:
# Instantiate the SRC layer
torch.manual_seed(123)
mla_layer_src = MultiHeadLatentAttention(d_model, num_heads, d_latent)

In [179]:
torch.manual_seed(123)
mla_layer_trgt = MultiLatentAttention(d_model, d_latent, num_heads)

In [180]:
# Create a dummy input tensor
dummy_input = torch.randn(batch_size, seq_len, d_model)

In [181]:
output_src = mla_layer_src(dummy_input)

In [182]:
output_trgt = mla_layer_trgt(dummy_input)

In [183]:
output_src.shape

torch.Size([4, 64, 512])

In [184]:
output_trgt.shape

torch.Size([4, 64, 512])

In [185]:
output_src[0,:4,:5]

tensor([[-0.0742, -0.0542, -0.2175,  0.2186, -0.0179],
        [-0.1800,  0.1269, -0.1347,  0.0531,  0.1250],
        [-0.0628,  0.0280, -0.1025,  0.0436,  0.1149],
        [-0.0123, -0.0085, -0.0912, -0.1126,  0.0263]],
       grad_fn=<SliceBackward0>)

In [186]:
output_trgt[0,:4,:5]

tensor([[-0.1466, -0.1084, -0.1542, -0.3744, -0.2573],
        [-0.1630, -0.1371, -0.1852, -0.2897, -0.0235],
        [-0.1615, -0.2075, -0.2433, -0.1965, -0.0224],
        [-0.2523, -0.1769, -0.2116, -0.1388, -0.0453]],
       grad_fn=<SliceBackward0>)

In [187]:
  torch.manual_seed(123)
  mla_layer_src = MultiHeadLatentAttention(d_model, num_heads, d_latent)

  mla_layer_trgt = MultiLatentAttention(d_model, d_latent, num_heads)
  mla_layer_trgt.WQ.load_state_dict(mla_layer_src.W_q.state_dict())
  mla_layer_trgt.WDKV.load_state_dict(mla_layer_src.W_dkv.state_dict())
  mla_layer_trgt.WDK.load_state_dict(mla_layer_src.W_uk.state_dict())
  mla_layer_trgt.WDV.load_state_dict(mla_layer_src.W_uv.state_dict())
  mla_layer_trgt.out_proj.load_state_dict(mla_layer_src.W_o.state_dict())

  torch.testing.assert_close(mla_layer_src(dummy_input), mla_layer_trgt(dummy_input))


## ROPE Positional Embeddings

In [ ]:

#
# LISTING 3.2: Building the Fused MLA and Decoupled RoPE Module
#
import torch
import torch.nn as nn
import math

class RotaryPositionalEncoding(nn.Module):
    """
    Helper module to apply Rotary Positional Encoding (RoPE).
    This is not added to the embeddings but is applied directly to
    the Query and Key vectors.
    """
    def __init__(self, d_head, max_seq_len=2048):
        super().__init__()
        # Precompute the theta values for the rotational matrix
        theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer('theta', theta)

        # Precompute the frequency terms (m * theta) for all positions
        positions = torch.arange(max_seq_len).unsqueeze(1)
        freqs = positions * self.theta.unsqueeze(0)

        # Create the complex number representation for rotation
        # The real part is cos(freqs) and the imaginary part is sin(freqs)
        self.register_buffer('freqs_cis', torch.polar(torch.ones_like(freqs), freqs))

    def forward(self, x, offset: int = 0):
        # x shape: (batch, num_heads, seq_len, d_head)
        # offset: absolute starting position of x in the full sequence.
        # During incremental decoding the new chunk lives at positions
        # [offset, offset + seq_len), not [0, seq_len).
        seq_len = x.shape[2]
        assert offset + seq_len <= self.freqs_cis.shape[0], \
            f"offset+seq_len ({offset + seq_len}) exceeds max_seq_len ({self.freqs_cis.shape[0]})"

        # Reshape x to treat pairs of dimensions as complex numbers
        x_complex = x.float().reshape(*x.shape[:-1], -1, 2)
        # Convert to PyTorch complex type
        x_complex = torch.view_as_complex(x_complex)

        # Slice precomputed frequencies at the absolute positions of x
        freqs_cis = self.freqs_cis[offset : offset + seq_len, :].unsqueeze(0).unsqueeze(0)

        # Apply rotation by multiplying in the complex domain
        # This rotates each pair of dimensions by the angle m * theta_i
        x_rotated = x_complex * freqs_cis

        # Convert back to real number representation
        x_rotated = torch.view_as_real(x_rotated)
        # Reshape back to the original d_head dimension
        x_rotated = x_rotated.flatten(3)

        return x_rotated.type_as(x)

In [189]:
d_head = 8
max_seq_len = 3

In [190]:
torch.arange(0, d_head, 2)

tensor([0, 2, 4, 6])

In [191]:
d_head = 8
theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))

In [192]:
theta

tensor([1.0000, 0.1000, 0.0100, 0.0010])

In [193]:
positions = torch.arange(max_seq_len).unsqueeze(1)

In [194]:
positions

tensor([[0],
        [1],
        [2]])

In [195]:
freqs = positions * theta.unsqueeze(0)
freqs

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03]])

In [196]:
freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
freqs_cis

tensor([[ 1.0000+0.0000j,  1.0000+0.0000j,  1.0000+0.0000j,  1.0000+0.0000j],
        [ 0.5403+0.8415j,  0.9950+0.0998j,  0.9999+0.0100j,  1.0000+0.0010j],
        [-0.4161+0.9093j,  0.9801+0.1987j,  0.9998+0.0200j,  1.0000+0.0020j]])

In [197]:
x = torch.randn((2,3,8))
x[0]

tensor([[ 0.0215,  0.8021,  0.1406, -0.4399,  0.9759,  0.5855,  1.3161, -0.3352],
        [-0.3336,  1.2017, -0.6480,  0.8064,  0.5367,  1.2833,  0.9036, -0.9156],
        [-0.4808, -0.7081,  0.6267, -0.5672, -0.5579,  0.1180,  2.3408, -0.6784]])

In [198]:
x_complex = x.float().reshape(*x.shape[:-1], -1, 2)
x_complex

tensor([[[[ 0.0215,  0.8021],
          [ 0.1406, -0.4399],
          [ 0.9759,  0.5855],
          [ 1.3161, -0.3352]],

         [[-0.3336,  1.2017],
          [-0.6480,  0.8064],
          [ 0.5367,  1.2833],
          [ 0.9036, -0.9156]],

         [[-0.4808, -0.7081],
          [ 0.6267, -0.5672],
          [-0.5579,  0.1180],
          [ 2.3408, -0.6784]]],


        [[[ 1.8622, -0.3391],
          [ 1.3476,  1.0728],
          [ 0.4266, -0.4726],
          [-0.2602,  0.2153]],

         [[-1.3106, -0.9346],
          [-0.1213,  2.3373],
          [-1.1059, -1.4287],
          [ 2.5270, -0.8644]],

         [[ 0.0858,  0.7694],
          [-1.4183, -0.8449],
          [ 1.0818,  0.4459],
          [-0.8705,  1.0443]]]])

In [199]:
x_complex = torch.view_as_complex(x_complex)
x_complex

tensor([[[ 0.0215+0.8021j,  0.1406-0.4399j,  0.9759+0.5855j,  1.3161-0.3352j],
         [-0.3336+1.2017j, -0.6480+0.8064j,  0.5367+1.2833j,  0.9036-0.9156j],
         [-0.4808-0.7081j,  0.6267-0.5672j, -0.5579+0.1180j,  2.3408-0.6784j]],

        [[ 1.8622-0.3391j,  1.3476+1.0728j,  0.4266-0.4726j, -0.2602+0.2153j],
         [-1.3106-0.9346j, -0.1213+2.3373j, -1.1059-1.4287j,  2.5270-0.8644j],
         [ 0.0858+0.7694j, -1.4183-0.8449j,  1.0818+0.4459j, -0.8705+1.0443j]]])

In [200]:
  freqs_cis = freqs_cis[:seq_len, :].unsqueeze(0).unsqueeze(0)
  freqs_cis.shape

torch.Size([1, 1, 3, 4])

In [201]:
x_complex.shape

torch.Size([2, 3, 4])

In [202]:
x_rotated = x_complex * freqs_cis
x_rotated.shape

torch.Size([1, 2, 3, 4])

In [203]:
x_rotated = torch.view_as_real(x_rotated)
x_rotated.shape

torch.Size([1, 2, 3, 4, 2])

In [204]:
x_rotated = x_rotated.flatten(3)
x_rotated.shape

torch.Size([1, 2, 3, 8])

In [205]:
x_rotated.type_as(x)



tensor([[[[ 0.0215,  0.8021,  0.1406, -0.4399,  0.9759,  0.5855,  1.3161,
           -0.3352],
          [-1.1914,  0.3685, -0.7253,  0.7377,  0.5238,  1.2886,  0.9045,
           -0.9147],
          [ 0.8440, -0.1425,  0.7269, -0.4314, -0.5601,  0.1068,  2.3421,
           -0.6737]],

         [[ 1.8622, -0.3391,  1.3476,  1.0728,  0.4266, -0.4726, -0.2602,
            0.2153],
          [ 0.0783, -1.6078, -0.3540,  2.3135, -1.0915, -1.4397,  2.5279,
           -0.8619],
          [-0.7353, -0.2421, -1.2222, -1.1098,  1.0727,  0.4675, -0.8726,
            1.0426]]]])

In [ ]:
class MultiLatentAttentionWithRope(nn.Module):
    def __init__(self, embed_dim, latent_dim, num_heads, max_seq_len = 1024, dropout=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.WDKV = nn.Linear(embed_dim, latent_dim)
        self.WUK = nn.Linear(latent_dim, embed_dim)
        self.WUV = nn.Linear(latent_dim, embed_dim)

        self.WKR = nn.Linear(embed_dim, latent_dim)

        self.WDQ = nn.Linear(embed_dim, latent_dim)
        self.WUQ = nn.Linear(latent_dim, embed_dim)
        self.WQR = nn.Linear(latent_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        self.rope = RotaryPositionalEncoding(self.head_dim, max_seq_len)

        self.register_buffer("mask", torch.triu(torch.ones(1,1, max_seq_len, max_seq_len), diagonal = 1).bool(), persistent = False)
        self.register_buffer("cache_udkv", torch.empty(0), persistent = False)
        # cache_kr now stores ALREADY-ROTATED KR of shape (batch, num_heads, seq, head_dim)
        self.register_buffer("cache_kr", torch.empty(0), persistent = False)
        self.ptr_current_seq = 0

    def forward(self, X, use_cache = False):
        batch_size, seq_len, embed_dim = X.size()
        initial_pos = self.ptr_current_seq

        CQ = self.WDQ(X)
        QC = self.WUQ(CQ)
        QR = self.WQR(CQ)

        UDKV = self.WDKV(X)
        KR_new = self.WKR(X)  # only the new tokens, shape (batch, seq_len, latent_dim)

        if use_cache:
            self.cache_udkv = torch.cat([self.cache_udkv, UDKV], dim = 1)
            UDKV = self.cache_udkv
        self.ptr_current_seq += seq_len

        KC = self.WUK(UDKV)
        V  = self.WUV(UDKV)

        # --- Decoupled RoPE: rotate ONLY the new KR at its absolute position ---
        # Expand shared latent KR_new across heads, reshape to (batch, num_heads, seq_len, head_dim)
        KR_new = KR_new.repeat(1, 1, embed_dim // self.latent_dim)
        KR_new = KR_new.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # Rotate the new tokens at their true absolute positions [initial_pos, initial_pos+seq_len)
        KR_new = self.rope(KR_new, offset=initial_pos)

        if use_cache:
            # Append PRE-ROTATED KR to the cache along the seq dim (dim=2 after transpose)
            if self.cache_kr.numel() == 0:
                self.cache_kr = KR_new
            else:
                self.cache_kr = torch.cat([self.cache_kr, KR_new], dim = 2)
            KR = self.cache_kr
        else:
            KR = KR_new

        # --- Q-side: reshape and rotate at the same offset ---
        QC = QC.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        QR = QR.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        KC = KC.view(batch_size, self.ptr_current_seq, self.num_heads, self.head_dim).transpose(1, 2)
        V  = V.view(batch_size, self.ptr_current_seq, self.num_heads, self.head_dim).transpose(1, 2)

        QR = self.rope(QR, offset=initial_pos)

        # attn_scores = (QC * KC.T + QR * KR.T)  / sqrt(head_dim)
        attn_scores_content = torch.matmul(QC, KC.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores_pos     = torch.matmul(QR, KR.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores = attn_scores_content + attn_scores_pos

        attn_scores = attn_scores.masked_fill(
                        self.mask[:, :, initial_pos : initial_pos + seq_len, :initial_pos + seq_len], float('-inf')
                        )
        attn_weights = nn.functional.softmax(attn_scores, dim = -1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, seq_len, embed_dim)
        context = self.out_proj(context)

        return context

### Verify RoPE offset: prefill vs incremental decode must match

If the offset is wired correctly, processing a full sequence in one shot must give
the same output as feeding it one token at a time with `use_cache=True`.
Without the offset, the incremental version rotates every new token as if it were
at position 0, and the two outputs diverge starting at token 1.

In [ ]:
torch.manual_seed(0)
embed_dim, latent_dim, num_heads = 512, 128, 8
T = 8
X_test = torch.randn(1, T, embed_dim)

layer = MultiLatentAttentionWithRope(embed_dim, latent_dim, num_heads)
layer.eval()

# Prefill: full sequence in one forward pass
with torch.no_grad():
    out_full = layer(X_test, use_cache=False)

# Incremental: feed one token at a time with caching enabled
layer.ptr_current_seq = 0
layer.cache_udkv = torch.empty(0)
layer.cache_kr   = torch.empty(0)

with torch.no_grad():
    outs = [layer(X_test[:, i:i+1, :], use_cache=True) for i in range(T)]
out_incremental = torch.cat(outs, dim=1)

torch.testing.assert_close(out_full, out_incremental, atol=1e-5, rtol=1e-5)
print("PASS: prefill and incremental decode produce identical outputs.")